[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ThanhDatVN/AI_journey/blob/main/notebooks/10_decision_trees/04_overfitting_and_pruning.ipynb)

# Bài 04 — Overfitting, pre-pruning và cost-complexity pruning

**🎯 Mục tiêu**
- Thấy cây không giới hạn overfit thế nào (train 100%, validation thấp).
- **Pre-pruning**: `max_depth`, `min_samples_leaf`, `min_samples_split`, `max_leaf_nodes`, `min_impurity_decrease` — chọn bằng CV.
- **Post-pruning**: cost-complexity pruning (`ccp_alpha`) — trồng cây đầy đủ rồi cắt bớt.
- Hiểu cây là model **variance cao**: dữ liệu thay đổi chút ít → cây khác hẳn (động lực cho Random Forest).

In [ ]:
# ⚙️ Setup — chạy được cả local lẫn Google Colab (local: đã `pip install -e .` thì bỏ qua)
import sys
if "google.colab" in sys.modules:
    !test -d /content/ml-course || git clone -q --depth 1 https://github.com/ThanhDatVN/AI_journey /content/ml-course
    !pip install -q -e /content/ml-course
    sys.path.insert(0, "/content/ml-course/src")
    %cd /content/ml-course/notebooks/10_decision_trees

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.model_selection import GridSearchCV, cross_val_score, train_test_split, validation_curve
from sklearn.tree import DecisionTreeClassifier, export_text

from ml_course.datasets import make_classification_2d, make_customer_churn
from ml_course.visualization import plot_decision_boundary, set_style

set_style()
churn = make_customer_churn(n=4000, seed=7)
X = pd.get_dummies(churn.drop(columns=["customer_id", "churn"]), dtype=float)
y = churn["churn"].to_numpy()
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=0, stratify=y)
print(X.shape, "| tỉ lệ churn:", y.mean().round(3))

## 1. Cây không giới hạn

In [ ]:
full = DecisionTreeClassifier(random_state=0).fit(X_train, y_train)
print(f"độ sâu = {full.get_depth()}, số lá = {full.get_n_leaves()} (trên {len(X_train)} mẫu train)")
print(f"accuracy train = {full.score(X_train, y_train):.3f} | test = {full.score(X_test, y_test):.3f}")
print(f"baseline (luôn đoán 'không churn') = {1 - y_test.mean():.3f}")

Trung bình mỗi lá chỉ chứa ~5 mẫu — cây "thuộc lòng" dữ liệu train, và test accuracy còn **thấp hơn** cả baseline "luôn đoán không churn"!

## 2. Pre-pruning: dừng sớm

| Hyperparameter | Ý nghĩa | Tăng/giảm → |
|---|---|---|
| `max_depth` | độ sâu tối đa | giảm → đơn giản hơn |
| `min_samples_split` | số mẫu tối thiểu để được chia | tăng → đơn giản hơn |
| `min_samples_leaf` | số mẫu tối thiểu ở mỗi lá | tăng → đơn giản hơn, xác suất ở lá mượt hơn |
| `max_leaf_nodes` | số lá tối đa (cây phát triển best-first) | giảm → đơn giản hơn |
| `min_impurity_decrease` | chỉ chia nếu gain ≥ ngưỡng | tăng → đơn giản hơn |

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4.2))
for ax, (param, values) in zip(axes, [("max_depth", np.arange(1, 21)), ("min_samples_leaf", [1, 2, 5, 10, 20, 40, 80, 160, 320])]):
    tr, va = validation_curve(DecisionTreeClassifier(random_state=0), X_train, y_train, param_name=param,
                              param_range=values, cv=5, scoring="roc_auc")
    ax.plot(values, tr.mean(1), "o-", label="train")
    ax.plot(values, va.mean(1), "o-", label="validation (5-fold CV)")
    ax.set_xlabel(param)
    ax.set_ylabel("ROC-AUC")
    ax.legend()
    if param == "min_samples_leaf":
        ax.set_xscale("log")
plt.suptitle("Validation curves cho cây quyết định", y=1.02)
plt.show()

In [ ]:
grid = GridSearchCV(DecisionTreeClassifier(random_state=0),
                    {"max_depth": [3, 4, 5, 6, 8, 10], "min_samples_leaf": [5, 20, 50, 100]},
                    cv=5, scoring="roc_auc").fit(X_train, y_train)
best_pre = grid.best_estimator_
print("tốt nhất:", grid.best_params_, f"| ROC-AUC CV = {grid.best_score_:.4f}")
print(f"số lá = {best_pre.get_n_leaves()} | accuracy test = {best_pre.score(X_test, y_test):.3f}")

## 3. Post-pruning: cost-complexity pruning

Vấn đề của pre-pruning: một split "trông vô ích" bây giờ có thể dẫn tới split rất tốt ở tầng dưới (vd XOR) — dừng sớm sẽ bỏ lỡ. **Post-pruning** trồng cây đầy đủ rồi **cắt ngược** những nhánh không đáng giá.

CART định nghĩa chi phí của cây $T$ với tham số $\alpha \ge 0$:

$$R_\alpha(T) = R(T) + \alpha\,|T|$$

trong đó $R(T)$ là tổng impurity (có trọng số) ở các lá, $|T|$ là số lá. Với mỗi $\alpha$ có một cây con tối ưu; $\alpha$ tăng → cây nhỏ dần. Thuật toán *weakest link pruning* liệt kê toàn bộ dãy $\alpha$ hiệu quả:

In [ ]:
path = DecisionTreeClassifier(random_state=0).cost_complexity_pruning_path(X_train, y_train)
alphas = path.ccp_alphas[:-1]                     # bỏ alpha cuối (cây chỉ còn gốc)
print(f"{len(alphas)} giá trị alpha hiệu quả, từ {alphas[0]:.2e} tới {alphas[-1]:.2e}")

alphas_grid = np.unique(np.quantile(alphas, np.linspace(0, 1, 30)))
cv_auc, n_leaves = [], []
for a in alphas_grid:
    model = DecisionTreeClassifier(random_state=0, ccp_alpha=a)
    cv_auc.append(cross_val_score(model, X_train, y_train, cv=5, scoring="roc_auc").mean())
    n_leaves.append(model.fit(X_train, y_train).get_n_leaves())
best_alpha = alphas_grid[int(np.argmax(cv_auc))]

fig, ax1 = plt.subplots(figsize=(9, 4))
ax1.semilogx(alphas_grid, cv_auc, "o-", label="ROC-AUC (CV)")
ax1.axvline(best_alpha, color="gray", ls="--")
ax1.set_xlabel("ccp_alpha")
ax1.set_ylabel("ROC-AUC (CV)")
ax2 = ax1.twinx()
ax2.semilogx(alphas_grid, n_leaves, "s--", color="tab:orange", alpha=0.6, label="số lá")
ax2.set_ylabel("số lá")
ax2.set_yscale("log")
plt.title(f"Cost-complexity pruning — alpha tốt nhất ≈ {best_alpha:.2e}")
plt.show()

pruned = DecisionTreeClassifier(random_state=0, ccp_alpha=best_alpha).fit(X_train, y_train)
print(f"cây đã cắt: {pruned.get_n_leaves()} lá, độ sâu {pruned.get_depth()} | accuracy test = {pruned.score(X_test, y_test):.3f}")

## 4. So sánh

In [ ]:
from sklearn.metrics import roc_auc_score

rows = []
for name, model in [("không giới hạn", full), ("pre-pruning (GridSearch)", best_pre), ("post-pruning (ccp_alpha)", pruned)]:
    rows.append({"cây": name, "số lá": model.get_n_leaves(), "độ sâu": model.get_depth(),
                 "accuracy test": model.score(X_test, y_test),
                 "ROC-AUC test": roc_auc_score(y_test, model.predict_proba(X_test)[:, 1])})
pd.DataFrame(rows).round(3)

Cây đã cắt **nhỏ hơn hàng chục lần** và tốt hơn hẳn trên test. ROC-AUC của cây không giới hạn rất thấp vì mỗi lá thuần → xác suất chỉ là 0 hoặc 1 (không xếp hạng được).

## 5. Variance cao: cây không ổn định

Train cây (độ sâu 4) trên các mẫu bootstrap khác nhau của cùng một dữ liệu — quan sát feature ở gốc và cấu trúc cây:

In [ ]:
rng = np.random.default_rng(0)
for b in range(6):
    idx = rng.choice(len(X_train), len(X_train), replace=True)
    t = DecisionTreeClassifier(max_depth=4, random_state=0).fit(X_train.iloc[idx], y_train[idx])
    first_two = export_text(t, feature_names=list(X.columns), max_depth=1).splitlines()[:2]
    print(f"bootstrap {b}: " + " | ".join(line.strip() for line in first_two))

In [ ]:
Xm, ym = make_classification_2d("moons", n=200, noise=0.35, seed=0)
fig, axes = plt.subplots(1, 4, figsize=(22, 4.4))
for ax, seed in zip(axes, range(4)):
    idx = np.random.default_rng(seed).choice(len(Xm), len(Xm), replace=True)
    t = DecisionTreeClassifier(random_state=0).fit(Xm[idx], ym[idx])
    plot_decision_boundary(t, Xm, ym, ax=ax)
    ax.set_title(f"bootstrap #{seed}")
plt.suptitle("Cùng dữ liệu, lấy mẫu lại → ranh giới khác hẳn (variance cao)", y=1.03)
plt.show()

Gốc luôn là `contract` (tín hiệu mạnh nhất), nhưng ngay split thứ hai đã nhảy giữa `tenure_months`, `support_calls`, `monthly_charges` với các ngưỡng khác nhau — và mọi split bên dưới thay đổi theo: một thay đổi nhỏ ở tầng trên **lan truyền** xuống toàn bộ cây con. Hình trên: với cây không giới hạn trên dữ liệu 2D, ranh giới khác hẳn nhau giữa các lần lấy mẫu. **Giải pháp:** trung bình hoá nhiều cây train trên các mẫu bootstrap → giảm variance mạnh → **Bagging / Random Forest** (chương 11).

## 🧠 Kiểm tra nhanh

1. Vì sao cây không giới hạn có accuracy train = 100% (khi không có hai mẫu trùng $\mathbf x$ khác nhãn)?
2. Pre-pruning và post-pruning khác nhau thế nào? Ưu điểm của post-pruning?
3. Tăng `ccp_alpha` thì cây thay đổi ra sao?
4. Vì sao ROC-AUC của cây không giới hạn thấp hơn hẳn cây đã cắt, dù accuracy chênh ít hơn?

## 📌 Tóm tắt

- Cây không giới hạn overfit nặng; luôn điều chỉnh độ phức tạp bằng CV.
- Pre-pruning (`max_depth`, `min_samples_leaf`...) đơn giản, nhanh; post-pruning (`ccp_alpha`) tránh dừng sớm sai.
- Cây có variance cao, không ổn định → nền tảng cho ensemble (chương 11–12).